# 03 · AI-2 fusion calibration

Fits the combined duplicate score (semantic + geospatial + temporal + category) on **synthetic pairs**, with non-negative weights. Lexical mode needs no credentials. Optional **provider-embedding mode** re-fits the weights on real embeddings (needs Internet + Kaggle Secret `OPENAI_API_KEY` + a configured embedding model).

> **All data in this notebook is SYNTHETIC** (template-generated). Metrics are a regression/ablation signal on unseen template families — **not** an estimate of real-world accuracy. Hindi/Marathi text needs native-speaker review.

In [ ]:
# --- 0. Locate the CivicConnect repo and set up a work directory (works on Kaggle and locally) ---------------
import importlib.util
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

def find_repo() -> Path:
    """Return a writable directory that contains ai/inference/schemas.py."""
    marker = Path("ai/inference/schemas.py")
    if os.environ.get("CIVIC_REPO"):                                   # explicit override
        return Path(os.environ["CIVIC_REPO"])
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:                                    # running from inside the repo
        if (p / marker).exists():
            return p
    if ON_KAGGLE:                                                      # attached Kaggle Dataset containing the repo or just ai/
        for hit in sorted(Path("/kaggle/input").glob("**/ai/inference/schemas.py"))[:1]:
            src = hit.parents[2]                                       # .../ai/inference/schemas.py -> parent of ai/
            dst = Path("/kaggle/working/repo")
            if not dst.exists():
                shutil.copytree(src, dst)                              # /kaggle/input is read-only
            return dst
        url = os.environ.get("CIVIC_GIT_URL", "https://github.com/tanujb03/CivicConnect_v2")                          # default: clone the project repo's `tanuj` branch (needs Internet ON); override with CIVIC_GIT_URL / CIVIC_GIT_REF
        if url:
            subprocess.run(["git", "clone", "--depth", "1", "--branch", os.environ.get("CIVIC_GIT_REF", "tanuj"), url, "/kaggle/working/repo"], check=True)
            return Path("/kaggle/working/repo")
    raise SystemExit("Cannot find the repo. On Kaggle: turn Internet ON (the notebook clones the project from GitHub) or add a Dataset containing the repo's `ai/` folder (see ai/training/README.md).")

REPO = find_repo()
sys.path.insert(0, str(REPO))
WORK = Path(os.environ.get("CIVIC_WORKDIR") or ("/kaggle/working" if ON_KAGGLE else tempfile.mkdtemp(prefix="civic_work_")))
WORK.mkdir(parents=True, exist_ok=True)

_MODULE = {"scikit-learn": "sklearn", "sentence-transformers": "sentence_transformers"}

def ensure(*pkgs):
    """pip install only what is missing (Kaggle images already ship numpy/scipy/scikit-learn/pydantic/httpx)."""
    missing = [p for p in pkgs if importlib.util.find_spec(_MODULE.get(p, p)) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

ensure("numpy", "scipy", "scikit-learn", "pydantic", "httpx")
print(f"repo : {REPO}\nwork : {WORK}\npython {sys.version.split()[0]} | kaggle={ON_KAGGLE}")

In [ ]:

from ai.evaluation import eval_fusion, run_eval
from ai.inference.fusion.scoring import FusionWeights
from ai.training.src import build_dataset, train_fusion_calibrator

DATA = WORK / "synthetic_v1"
if not (DATA / "pairs_train.jsonl").exists():
    build_dataset.main(["--out", str(DATA)])
pairs = run_eval.read_jsonl(REPO / "ai/evaluation/datasets/fusion_eval_pairs.v1.jsonl")

OUT_LEX = train_fusion_calibrator.run(DATA, WORK / "artifacts" / "fusion_calibrator")
res = eval_fusion.evaluate(pairs, FusionWeights.load(OUT_LEX))
print(f"\nheld-out pairs: AUC={res['auc_roc']}  at policy threshold: {res['at_possible_duplicate_threshold']}")
print("same-language:", res.get("same_language"), "| cross-language:", res.get("cross_language"))

### Optional: provider-embedding mode
Skipped automatically without credentials. Never print or commit the key. Add `OPENAI_API_KEY` under *Add-ons → Secrets*, set the embedding model ID below (validate it against the provider's current catalogue first).

In [ ]:
EMBED_MODEL = os.environ.get("AI_EMBEDDING_MODEL", "")      # e.g. set to the embedding model ID you validated; no default on purpose
if ON_KAGGLE and not os.environ.get("OPENAI_API_KEY"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["OPENAI_API_KEY"] = UserSecretsClient().get_secret("OPENAI_API_KEY")
    except Exception as e:                                    # secret not attached
        print("no Kaggle secret OPENAI_API_KEY:", type(e).__name__)

if os.environ.get("OPENAI_API_KEY") and EMBED_MODEL:
    os.environ["AI_EMBEDDING_MODEL"] = EMBED_MODEL
    OUT_EMB = train_fusion_calibrator.run(DATA, WORK / "artifacts" / "fusion_calibrator_embedding", semantic_mode="provider")
    print("embedding-mode weights:", OUT_EMB)
else:
    OUT_EMB = None
    print("SKIPPED provider-embedding calibration: needs OPENAI_API_KEY (Kaggle Secret) and EMBED_MODEL. Lexical weights above are still valid.")

In [ ]:
import zipfile

BUNDLE = WORK / "civic_fusion_weights.zip"
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as z:
    for out in [o for o in (OUT_LEX, OUT_EMB) if o]:
        for f in sorted(out.iterdir()):
            z.write(f, f"{out.parent.name}/{out.name}/{f.name}")
print("download:", BUNDLE)